# Week 102, Day 4: Review: engineering

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w102d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('fares.py', 'w', encoding='utf-8') as f:
    f.write('def fare(km, age, member=False):\n    """The fare for a minibus seat: 50p a km, half price under 16 or 65 and over, 10% off for members."""\n    if km <= 0:\n        raise ValueError("km must be positive")\n    price = round(km * 50)\n    if age < 16 or age >= 65:\n        price = price // 2\n    if member:\n        price = price * 9 // 10\n    return price\n')

**Predict before you run:** what will this print?



In [ ]:
from urllib.parse import urlsplit, parse_qs, urlencode

url = urlsplit("https://waymark.example/v1/walks?month=2026-02&limit=5&tag=hilly&tag=dogs")
print(url.path, parse_qs(url.query)["tag"], urlencode({"q": "Mill & Weir", "page": 2}))

In [ ]:
from fares import fare

cases = [(10, 30, False, 500), (10, 10, False, 250), (10, 65, False, 250), (10, 64, False, 500), (10, 30, True, 450), (3, 70, True, 67)]
print([fare(*c[:3]) == c[3] for c in cases])

In [ ]:
raw = "POST /v1/walks?draft=1 HTTP/1.1\r\nHost: waymark.example\r\nContent-Type: application/json\r\n\r\n{\"km\": 6.0}"
head, body = raw.split("\r\n\r\n", 1)
request_line, *header_lines = head.split("\r\n")
method, target, version = request_line.split(" ")
headers = {k.lower(): v.strip() for k, v in (h.split(":", 1) for h in header_lines)}
print(method, target, headers["content-type"], body)

In [ ]:
import hmac, hashlib

key = b"waymark-demo-key"
payload = b'{"route": "Ridge walk", "user": 7}'
signature = hmac.new(key, payload, hashlib.sha256).hexdigest()
tampered = payload.replace(b"7", b"8")
print(len(signature), hmac.compare_digest(signature, hmac.new(key, tampered, hashlib.sha256).hexdigest()))

> **Key idea.** Tests: parametrize, fixtures, raises, approx, mocks, mutants. Packages: pyproject, semver, bounded dependencies. HTTP: method, path and query, case-insensitive headers, body; the right status codes. Route by method and path; validate JSON. SQL with parameters, in transactions, with aggregates. Security: `secrets`, `compare_digest`, HMAC, escaping. Design with estimates and trade-offs.

### Exercise: Tests that catch mistakes

Write pytest tests for `fare` in `fares.py` that pass on it and catch all four broken versions: the age boundary at 65 wrong, the age boundary at 16 wrong, the member discount applied before the age discount (giving a different rounding), and no error for a distance of 0.

In [ ]:
import pytest
from fares import fare

def test_adult():
    assert fare(10, 30) == 500

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"fares.py\").read()\nBROKEN = {\n    \"65 counted as adult\": GOOD.replace(\"age >= 65\", \"age > 65\"),\n    \"16 counted as child\": GOOD.replace(\"age < 16\", \"age <= 16\"),\n    \"discounts swapped\": GOOD.replace('''    if age < 16 or age >= 65:\n        price = price // 2\n    if member:\n        price = price * 9 // 10''', '''    if member:\n        price = price * 9 // 10\n    if age < 16 or age >= 65:\n        price = price // 2'''),\n    \"zero allowed\": GOOD.replace(\"if km <= 0:\", \"if km < 0:\"),\n}\n\ndef test_suite():\n    \"all four caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"fares\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"fares.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Reading an HTTP request

Write `parse_request(raw)`: split the head from the body at the first blank line (`\r\n\r\n`), the request line into method, target and version, and the headers into a dict with lower-case names and stripped values. Return a dict with `method`, `path`, `query` (from `parse_qs`, each value a list), `headers` and `body` (a string, `""` if none).

In [ ]:
from urllib.parse import parse_qs, urlsplit

def parse_request(raw):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("RAW = \"POST /v1/walks?draft=1&tag=a&tag=b HTTP/1.1\\r\\nHost: waymark.example\\r\\nContent-Type:  application/json\\r\\n\\r\\n{\\\"km\\\": 6.0}\"\n\ndef test_request():\n    \"method, path, query, headers, body\"\n    r = parse_request(RAW)\n    assert r[\"method\"] == \"POST\" and r[\"path\"] == \"/v1/walks\" and r[\"query\"] == {\"draft\": [\"1\"], \"tag\": [\"a\", \"b\"]}\n    assert r[\"headers\"] == {\"host\": \"waymark.example\", \"content-type\": \"application/json\"} and r[\"body\"] == '{\"km\": 6.0}'\n    g = parse_request(\"GET / HTTP/1.1\\r\\nHost: x\\r\\n\\r\\n\")\n    assert g[\"path\"] == \"/\" and g[\"query\"] == {} and g[\"body\"] == \"\"")

### Exercise: Routes with parameters

Write a class `Router` with `add(method, pattern, handler)` where patterns look like `/v1/walks/<int:walk_id>` or `/v1/users/<name>`, and `dispatch(method, path)`: call the matching handler with the parameters as keyword arguments (ints converted) and return its result; return `(404, {"error": "not found"})` if no pattern matches the path, or `(405, {"error": "method not allowed"})` if a pattern matches but not for this method.

In [ ]:
import re

class Router:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_router():\n    \"patterns, 404 and 405\"\n    r = Router()\n    r.add(\"GET\", \"/v1/walks/<int:walk_id>\", lambda walk_id: (200, {\"id\": walk_id}))\n    r.add(\"DELETE\", \"/v1/walks/<int:walk_id>\", lambda walk_id: (204, None))\n    r.add(\"GET\", \"/v1/users/<name>/walks\", lambda name: (200, {\"user\": name}))\n    assert r.dispatch(\"GET\", \"/v1/walks/17\") == (200, {\"id\": 17}) and r.dispatch(\"DELETE\", \"/v1/walks/3\") == (204, None)\n    assert r.dispatch(\"GET\", \"/v1/users/ada/walks\") == (200, {\"user\": \"ada\"})\n    assert r.dispatch(\"PUT\", \"/v1/walks/17\") == (405, {\"error\": \"method not allowed\"})\n    assert r.dispatch(\"GET\", \"/v1/walks/abc\") == (404, {\"error\": \"not found\"}) and r.dispatch(\"GET\", \"/v1/nope\") == (404, {\"error\": \"not found\"})")

### Exercise: A query with parameters

Write `monthly_km(conn, user)`: from a SQLite table `walk(user, day, km)` (day as `YYYY-MM-DD`), return `(month, total)` pairs for one user, months ascending, totals rounded to 1 place, using a parameter for the user, `substr(day, 1, 7)` and `GROUP BY`.

In [ ]:
import sqlite3

def monthly_km(conn, user):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\ndef test_monthly():\n    \"parameters, aggregates\"\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE walk (user TEXT, day TEXT, km REAL)\")\n    conn.executemany(\"INSERT INTO walk VALUES (?, ?, ?)\", [(\"ada\", \"2026-01-04\", 6.0), (\"ada\", \"2026-01-18\", 7.5), (\"bo\", \"2026-01-05\", 3.0),\n                                                           (\"ada\", \"2026-02-01\", 11.0), (\"o'brien\", \"2026-02-02\", 4.0)])\n    assert monthly_km(conn, \"ada\") == [(\"2026-01\", 13.5), (\"2026-02\", 11.0)]\n    assert monthly_km(conn, \"o'brien\") == [(\"2026-02\", 4.0)] and monthly_km(conn, \"x' OR '1'='1\") == []")

### Exercise: A link that can't be edited

Write `sign(data, key)`: return `"PAYLOAD.SIGNATURE"` where PAYLOAD is URL-safe base64 of the JSON of `data` (sorted keys, no padding) and SIGNATURE the hex HMAC-SHA256 of PAYLOAD with `key` (bytes). Then `unsign(token, key)`: the data, or `None` if the token is malformed or the signature doesn't match (compare with `hmac.compare_digest`).

In [ ]:
import base64
import hashlib
import hmac
import json

def sign(data, key):
    ___

def unsign(token, key):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_signed():\n    \"tamper-evident\"\n    key = b\"demo-\" + b\"key-for-tests\"\n    token = sign({\"route\": \"Ridge walk\", \"user\": 7}, key)\n    assert unsign(token, key) == {\"route\": \"Ridge walk\", \"user\": 7}\n    payload, sig = token.split(\".\")\n    import base64, json\n    forged = base64.urlsafe_b64encode(json.dumps({\"route\": \"Ridge walk\", \"user\": 8}, sort_keys=True).encode()).decode().rstrip(\"=\")\n    assert unsign(forged + \".\" + sig, key) is None and unsign(token, b\"other\") is None and unsign(\"nonsense\", key) is None\n    assert \"=\" not in payload")

### Bug hunt: A query that broke on an ampersand

Joining the query by hand leaves `&` and spaces in the value, so the server sees three parameters instead of two and this prints `3`. Encode it with `urlencode` so it prints `2`.

In [ ]:
from urllib.parse import parse_qs, urlencode

params = {"q": "Mill & Weir", "page": 2}
query = "&".join(f"{k}={v}" for k, v in params.items())
print(len(query.split("&")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"encode values\"\n    expect_output(\"2\")")

### Bug hunt: Created, but said OK

A `POST` that creates a walk should answer `201 Created`, but this answers `200`. Fix the status so it prints `201 {'id': 1}`.

In [ ]:
walks = {}

def create(body):
    walk_id = len(walks) + 1
    walks[walk_id] = body
    return 200, {"id": walk_id}

status, data = create({"km": 6.0})
print(status, data)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the right status\"\n    expect_output(\"201 {'id': 1}\")")

### Puzzle: A safe query

Arrange `monthly_km`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status for a request whose method isn't allowed on that path
q = ___   # parse_qs("tag=a&tag=b")
r = ___   # the comparison function for signatures (module.function)
s = ___   # True or False: SQL parameters protect against injection
t = ___   # fare(10, 65)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 405 and q == {\"tag\": [\"a\", \"b\"]} and r == \"hmac.compare_digest\", \"p, q and r\"\n    assert s is True and t == 250, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a walks service in memory

Write `WalksService` with `handle(method, path, body=None)` returning `(status, data)`, using a router of your own (or Day 4's): `POST /v1/walks` with a JSON body `{"km": number > 0, "route": non-empty string}` creates a walk and returns `201` and `{"id": N, "km": ..., "route": ...}` (ids from 1); invalid JSON gives `400 {"error": "invalid JSON"}`, invalid fields `422 {"error": "km must be positive"}` or `422 {"error": "route is required"}`; `GET /v1/walks/<id>` returns `200` and the walk or `404 {"error": "not found"}`; `GET /v1/walks` returns `200` and `{"walks": [...], "total_km": T}` (T to 1 place); `DELETE /v1/walks/<id>` returns `204` and `None`, or `404`; any other method on those paths `405 {"error": "method not allowed"}`; other paths `404`.

In [ ]:
import json
import re

class WalksService:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_service():\n    \"a backend, end to end\"\n    s = WalksService()\n    assert s.handle(\"POST\", \"/v1/walks\", '{\"km\": 6.0, \"route\": \"Ridge walk\"}') == (201, {\"id\": 1, \"km\": 6.0, \"route\": \"Ridge walk\"})\n    assert s.handle(\"POST\", \"/v1/walks\", '{\"km\": 3.5, \"route\": \"Lake loop\"}')[0] == 201\n    assert s.handle(\"POST\", \"/v1/walks\", \"{not json\") == (400, {\"error\": \"invalid JSON\"})\n    assert s.handle(\"POST\", \"/v1/walks\", '{\"km\": 0, \"route\": \"x\"}') == (422, {\"error\": \"km must be positive\"})\n    assert s.handle(\"POST\", \"/v1/walks\", '{\"km\": 2, \"route\": \"\"}') == (422, {\"error\": \"route is required\"})\n    assert s.handle(\"GET\", \"/v1/walks/2\") == (200, {\"id\": 2, \"km\": 3.5, \"route\": \"Lake loop\"})\n    assert s.handle(\"GET\", \"/v1/walks\") == (200, {\"walks\": [{\"id\": 1, \"km\": 6.0, \"route\": \"Ridge walk\"}, {\"id\": 2, \"km\": 3.5, \"route\": \"Lake loop\"}], \"total_km\": 9.5})\n    assert s.handle(\"DELETE\", \"/v1/walks/1\") == (204, None) and s.handle(\"DELETE\", \"/v1/walks/1\") == (404, {\"error\": \"not found\"})\n    assert s.handle(\"PUT\", \"/v1/walks/2\") == (405, {\"error\": \"method not allowed\"}) and s.handle(\"GET\", \"/v2/x\") == (404, {\"error\": \"not found\"})\n    assert s.handle(\"POST\", \"/v1/walks\", '{\"km\": true, \"route\": \"x\"}') == (422, {\"error\": \"km must be positive\"})")

## Warm-up

URLs, queries and their encoding are where web bugs start: `&` and spaces inside a value must be encoded, and a query can repeat a key. The standard library gets all of it right, which is a recurring theme of today: use the tool that already handles the edge cases.

## Tests and packages

pytest: plain `assert`, `@pytest.mark.parametrize` for many cases, fixtures for setup (with `yield` for clean-up), `pytest.raises` for errors, `pytest.approx` for floats, mocks and fakes for slow or external things, and properties for invariants. A good suite fails on plausible mistakes (mutants). Packages: `pyproject.toml` with name, version (semver), dependencies with bounds, and scripts; builds give an sdist and a wheel.

## HTTP and the web

A request has a method, a path with a query, headers (case-insensitive names) and an optional body; a response has a status, headers and a body. Status codes: 200 OK, 201 Created, 204 No Content, 400 Bad Request, 401/403 for authentication and permission, 404 Not Found, 405 Method Not Allowed, 409 Conflict, 422 for invalid content, 500 for server errors. A web backend routes method and path to a handler, parses and validates the body (usually JSON), and returns JSON with the right status.

## Data, security and design

Databases: SQL with **parameters**, never string formatting; transactions so changes happen completely or not at all; `GROUP BY` and aggregates. Security: secrets from `secrets`, comparisons with `hmac.compare_digest`, HMAC signatures to detect tampering, escaping output, least privilege. Design: estimate, cache, queue, replicate, and say the trade-offs. Data: `csv`, `statistics`, and pandas for bigger tables.

## Practice

## Quiz

## Daily challenge

## Before you go

Tomorrow is the practice final: the real exam's format and timing, on a shorter paper, with model answers afterwards.